## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.normal(shape=(28*28, 128), stddev=0.01), trainable=True)
        self.b1 = tf.Variable(tf.zeros(shape=(128,)), trainable=True)
        self.W2 = tf.Variable(tf.random.normal(shape=(128,10), stddev=0.01), trainable=True)
        self.b2 = tf.Variable(tf.zeros(shape=(10,)), trainable=True)
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x_f = tf.reshape(x, [-1, 784])
        h1 = tf.matmul(x_f, self.W1) + self.b1
        act_h1 = tf.tanh(h1)
        logits = tf.matmul(act_h1, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.3027244 ; accuracy 0.08241667
epoch 1 : loss 2.3024347 ; accuracy 0.087016664
epoch 2 : loss 2.3021452 ; accuracy 0.09165
epoch 3 : loss 2.3018558 ; accuracy 0.09765
epoch 4 : loss 2.3015664 ; accuracy 0.103483334
epoch 5 : loss 2.3012772 ; accuracy 0.1093
epoch 6 : loss 2.300988 ; accuracy 0.11508334
epoch 7 : loss 2.3006988 ; accuracy 0.12165
epoch 8 : loss 2.3004093 ; accuracy 0.12838334
epoch 9 : loss 2.3001199 ; accuracy 0.13528334
epoch 10 : loss 2.2998307 ; accuracy 0.14238334
epoch 11 : loss 2.2995412 ; accuracy 0.14955
epoch 12 : loss 2.2992518 ; accuracy 0.15748334
epoch 13 : loss 2.2989619 ; accuracy 0.16493334
epoch 14 : loss 2.2986724 ; accuracy 0.17316666
epoch 15 : loss 2.298382 ; accuracy 0.18116666
epoch 16 : loss 2.2980921 ; accuracy 0.18955
epoch 17 : loss 2.2978015 ; accuracy 0.19843334
epoch 18 : loss 2.2975109 ; accuracy 0.20778333
epoch 19 : loss 2.2972198 ; accuracy 0.21718334
epoch 20 : loss 2.296929 ; accuracy 0.22686666
epoch 21 : loss 2.2966